In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import sys

sys.path.append('/home/wolfgang/repos/AirGo_Apnea_Detection')
from apnea_detection_functions import apply_apnea_prediction_models

sys.path.append('/home/wolfgang/repos/ICU-Sleep/code1')
from airgo_features import compute_airgo_features
from sleep_staging_functions import apply_airgo_sleep_staging_models

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import *
from sleep_analysis_functions import *
%matplotlib widget
import matplotlib.pyplot as plt

### Compute all airgo features (RR, MV, instability, complexity, actigraphy), apply apnea detection and sleep staging model. <br>  If other signals are available, like SpO2 (bedmaster, PSG), then models that use these signals will also be utilized.
Currently models:
Apnea Detection: Respiration Only (needs features computed), Respiration+SpO2.  
Sleep Staging: Based on Respiration Only, Actigraphy Only.

### input: 
<b> list of paths to files. </b>  
files: either original airgo files, or merged with bedmaster or PSG data.

In [2]:
data_dir = './Data_Analysis/Airgo'
output_dir = './Data_Analysis/Airgo_Features'

data_dir = '/media/ssd2/icu_final_files'
output_dir = '/media/ssd2/icu_sleepstaging_TMP2'

data_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data'
output_dir = '/media/ssd2/Combined_Data2'

overwrite = True

filepaths_input, filepaths_output = filepaths_routine(data_dir, output_dir)

Number of files:         10
Sample filepath input:   /media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data/cov_resp_001.h5
Sample filepath output:  /media/ssd2/Combined_Data2/cov_resp_001.h5


### Params / Settings

In [3]:
fs_manual = 10
do_resample_and_interpolation = False       # recommended for raw airgo, resampling to 'perfect 10Hz'
do_compute_airgo_features = False           # all features. by default, complexity features are computed in this code which are the slowest but needed for apnea prediction.
do_apply_sleep_staging_models = True        # respiration-only, actigraphy-only.
do_apply_apnea_prediction_models = True    # respiration-only and respiration+spo2 models. if sleep_stage_available, sleep-only apnea versions get computed too.
do_compute_self_similarity = True          # depends on airgo available
do_compute_hypoxia_burden = True           # depends on apnea predictions and sleep stages (for hours of sleep)


### compute_hypoxia_burden params ###
apnea_name = 'apnea_pred_va_a3'                      # name of Apnea info column
hours_sleep = 'stage_pred_amendsumeffort'  # name of sleep stage column, or int/float for manual setting.



In [ ]:
# filepath_input = filepaths_input[0]
# filepath_output = filepaths_output[0]

for filepath_input, filepath_output in list(zip(filepaths_input, filepaths_output)):

    try:
#     if 1:
        
        if (os.path.exists(filepath_output)) & (not overwrite):
            print(f'{filepath_output} already exists. Continue.')
            continue

        print(filepath_input)

        data, hdr, fs = read_in_routine(filepath_input, fs_manual=fs_manual)

#         print('TMP!!!!')
#         index_airgo_start = data.band.dropna().index[0]
#         index_airgo_later = data.band.dropna().index[fs*3600]
#         data = data.loc[index_airgo_start : index_airgo_later]
        
        # skip missing airgo:
        if not 'band' in data.columns:
            print('No AirGo data at all.')
            continue
            
        # skip missing airgo:
        if not 'band' in data.columns:
            print('No AirGo data at all.')
            continue        
        
#         if data.acc_ai_10sec.dropna().mean() > 1:
#             print(file_tmp)
#             print('data.acc_ai_10sec.dropna().mean() > 1 --> V4.')
#             continue
            
#         if any(data.band_unscaled.dropna()>5000):
#             print(file_tmp)
#             print('unscaled airgo amplitude>5000 --> V4.')
#             continue
                        
        
        if do_resample_and_interpolation:
            data = airgo_resample_preprocess(data)

        if do_compute_airgo_features:
            data = compute_airgo_features(data, fs=fs, complexity_features=1)

        if do_apply_sleep_staging_models:
            
#             data.drop(['stage_pred_amendsumeffort'], axis=1, inplace=True)
#             data.drop(['stage_pred_activity1sec'], axis=1, inplace=True)
#             data.drop(['stage_pred_vcomb1'], axis=1, inplace=True)
            
            data = apply_airgo_sleep_staging_models(data, fs=fs)
        
        if do_apply_apnea_prediction_models:
            data = apply_apnea_prediction_models(data, fs=fs)
            
        # to be tested:
        if do_compute_self_similarity:
            data = self_similarity_airgo(data)
            
        if do_compute_hypoxia_burden:
            data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, hours_sleep = hours_sleep)

        save_data_routine(data, filepath_output, hdr=hdr, overwrite=overwrite)
        
    except Exception as e:
        print('Exception for ' + filepath_input)
        print(e)
        continue

/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data/cov_resp_001.h5
amendsumeffort
activity10sec


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/torch/serialization.py:657: SourceChangeWarning: source code of class 'mymodel.CHESTSleepNet' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)


Following Apnea Models loaded: ['rsr_a3', 'ro_a3']
compute spo2 feature...


/home/wolfgang/repos/AirGo_Apnea_Detection/apnea_detection_functions.py:309: RuntimeWarning: All-NaN slice encountered
  spo2_drop_feature = [np.nanmax(rolling_spo2_drop_array[x - samples_pre: x + samples_post]) for x in


rsr_a3 not applied, no SpO2 available.


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/lib/nanfunctions.py:1391: RuntimeWarning: All-NaN slice encountered
  overwrite_input, interpolation)
/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:70: RuntimeWarning: Mean of empty slice
  mean_spo2 = np.nanmean(np.array(spo2_collection), axis=0)
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/core/_methods.py:217: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/core/_methods.py:186: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/core/_methods.py:209: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Exception for /media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data/cov_resp_001.h5
name 'filepath_input' is not defined
/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data/cov_resp_002.h5
amendsumeffort
activity10sec


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/torch/serialization.py:657: SourceChangeWarning: source code of class 'mymodel.CHESTSleepNet' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)


Following Apnea Models loaded: ['rsr_a3', 'ro_a3']
compute spo2 feature...


In [1]:
data

NameError: name 'data' is not defined

In [38]:
data2 = data[np.logical_not(pd.isna(data.band))]

In [36]:
# data2 = data2.iloc[fs*3600*140:]

In [40]:
data2

,cuff,hr,nbpd,nbpm,nbps,pvc,spo2,spo2r,stavf,stavl,...,ventilation_60s,ventilation_8s,ventilation_cvar_1min,ventilation_cvar_2min,ventilation_cvar_30sec,ventilation_cvar_5min,stage_pred_amendsumeffort,stage_pred_activity10sec,stage_pred_comb_breath_activity_1,to_predict_apnea
2020-04-15 18:21:21.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.776367,1.115234,0.440430,1.728516,NaN,NaN,NaN,0
2020-04-15 18:21:21.600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.777344,1.116211,0.444824,1.729492,NaN,NaN,NaN,0
2020-04-15 18:21:21.700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.778809,1.117188,0.449219,1.729492,NaN,NaN,NaN,0
2020-04-15 18:21:21.800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.779785,1.118164,0.453369,1.730469,NaN,NaN,NaN,0
2020-04-15 18:21:21.900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.780762,1.119141,0.457520,1.730469,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-04-21 08:15:39.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.054688,16.312500,0.272705,0.501465,0.165405,0.700195,NaN,NaN,NaN,1
2020-04-21 08:15:39.100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.070312,16.468750,0.273193,0.500488,0.165283,0.700195,NaN,NaN,NaN,0
2020-04-21 08:15:39.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.062500,16.468750,0.273438,0.499268,0.165161,0.700195,NaN,NaN,NaN,0
2020-04-21 08:15:39.300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.062500,16.484375,0.273926,0.498047,0.165161,0.700195,NaN,NaN,NaN,0


In [41]:
data.spo2.dropna()

2020-04-11 21:33:20.000    95.0
2020-04-11 21:33:20.100    95.0
2020-04-11 21:33:20.200    95.0
2020-04-11 21:33:20.300    95.0
2020-04-11 21:33:20.400    95.0
                           ... 
2020-04-12 16:30:33.600    93.0
2020-04-12 16:30:33.700    93.0
2020-04-12 16:30:33.800    93.0
2020-04-12 16:30:33.900    93.0
2020-04-12 16:30:34.000    93.0
Name: spo2, Length: 676076, dtype: float64

In [15]:
if do_apply_sleep_staging_models:

#             data.drop(['stage_pred_amendsumeffort'], axis=1, inplace=True)
#             data.drop(['stage_pred_activity1sec'], axis=1, inplace=True)
#             data.drop(['stage_pred_vcomb1'], axis=1, inplace=True)

    data2 = apply_airgo_sleep_staging_models(data2, fs=fs)

if do_apply_apnea_prediction_models:
    data2 = apply_apnea_prediction_models(data2, fs=fs)

# to be tested:
if do_compute_self_similarity:
    data2 = self_similarity_airgo(data2)

if do_compute_hypoxia_burden:
    data2, hypoxia_burden = compute_hypoxia_burden(data2, fs, apnea_name = apnea_name, hours_sleep = hours_sleep)


amendsumeffort
activity10sec
Following Apnea Models loaded: ['rsr_a3', 'ro_a3']
rsr_a3 not applied, no SpO2 available.


TypeError: unsupported operand type(s) for &: 'bool' and 'str'

In [16]:
data = apply_apnea_prediction_models(data, fs=fs)

Following Apnea Models loaded: ['rsr_a3', 'ro_a3']
rsr_a3 not applied, no SpO2 available.


In [21]:


# to be tested:
if do_compute_self_similarity:
    data = self_similarity_airgo(data, fs=fs, verbose=True)

if do_compute_hypoxia_burden:
    data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, hours_sleep = hours_sleep)


In [7]:
data.columns

Index(['cuff', 'hr', 'nbpd', 'nbpm', 'nbps', 'pvc', 'spo2', 'spo2r', 'stavf',
       'stavl',
       ...
       'ventilation_60s', 'ventilation_8s', 'ventilation_cvar_1min',
       'ventilation_cvar_2min', 'ventilation_cvar_30sec',
       'ventilation_cvar_5min', 'stage_pred_amendsumeffort',
       'stage_pred_activity10sec', 'stage_pred_comb_breath_activity_1',
       'to_predict_apnea'],
      dtype='object', length=109)

2020-04-15 18:21:21.500    0.0
2020-04-15 18:21:22.500    0.0
2020-04-15 18:21:23.500    0.0
2020-04-15 18:21:24.500    0.0
2020-04-15 18:21:25.500    0.0
                          ... 
2020-04-15 19:21:39.500    0.0
2020-04-15 19:21:40.500    0.0
2020-04-15 19:21:41.500    0.0
2020-04-15 19:21:42.500    0.0
2020-04-15 19:21:43.500    0.0
Name: apnea_pred_va_a3, Length: 3623, dtype: float64

In [28]:
plt.subplots(sharex=True, figsize=(9,9))
ax1 = plt.subplot(611)
ax1.plot(data.band)
ax2 = plt.subplot(612, sharex=ax1)
ax2.plot(data.apnea_pred_va_a3.dropna())
ax3 = plt.subplot(613, sharex=ax1)
ax3.plot(data.stage_pred_amendsumeffort)
ax4 = plt.subplot(614, sharex=ax1)
ax4.plot(data.self_similarity)
ax5 = plt.subplot(615, sharex=ax1)
ax5.plot(data.stage_pred_activity10sec)
ax5 = plt.subplot(616, sharex=ax1)
ax5.plot(data.acc_ai_10sec)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
# plt.subplots(3, sharex=True)
# ax1 = plt.subplot(511)
# ax1.plot(data.band)
# ax2 = plt.subplot(512, sharex=ax1)
# ax2.plot(data.apnea_prob_RO_a3.dropna())
# ax3 = plt.subplot(513, sharex=ax1)
# ax3.plot(data.stage_pred_amendsumeffort)
# ax4 = plt.subplot(514, sharex=ax1)
# ax4.plot(data.stage_pred_activity1sec)
# ax5 = plt.subplot(515, sharex=ax1)
# ax5.plot(data.acc_ai_1sec)

In [ ]:
data.columns